# M3: executor RL with poset vs chain vs flat credit (the real-LLM test of P6)

The planner is the frozen base model; plans are generated once and cached, so every method trains on **the same plans and problems in the same order**. Methods differ only in the per-step advantage:

* `flat`: outcome-only GRPO
* `chain`: a step is credited only if every step before it in the written order verified
* `poset`: a step is credited if it and its prerequisites verified (verified ideal)

Intermediate steps are verified by **outcome-anchored consensus** (agreement with rollouts whose final answer is correct). Evaluation is split by plan width: P6 predicts `poset` beats `chain` more on wider plans.

Single-GPU sizing (A100 80GB or G4): a 1.5B model with LoRA, 16 problems x 8 rollouts per step. Everything checkpoints to Drive and resumes.

In [ ]:
# 1. GPU check: L4 / A100 / G4 recommended (T4 works for M2 with a <=2B model)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Install (vLLM brings a matching PyTorch). Takes a few minutes; restart the runtime if Colab asks.
!pip install -q vllm peft datasets sympy antlr4-python3-runtime==4.11

In [ ]:
# 3. Mount Drive and unpack the code. Upload pohrl.zip to MyDrive/pohrl/ first.
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/pohrl'
!mkdir -p /content/code && cd /content/code && unzip -oq {BASE}/pohrl.zip
import sys; sys.path.insert(0, '/content/code/pohrl')
import pohrl, pohrl_llm; print('code ok')

In [ ]:
# 4. Configuration
import torch
MODEL = 'Qwen/Qwen2.5-Math-1.5B-Instruct'   # or 'Qwen/Qwen3-1.7B'
DTYPE = 'bfloat16' if torch.cuda.get_device_capability()[0] >= 8 else 'float16'
METHODS = ['poset', 'chain', 'flat']
SEEDS = [0]                       # add 1, 2 when budget allows
STEPS, PROBLEMS_PER_STEP, G = 150, 16, 8
EVAL_EVERY, N_TRAIN, N_EVAL = 25, 2000, 150
OUT = f"{BASE}/m3_{MODEL.split('/')[-1]}"

In [ ]:
# 5. Data: MATH train (levels 3-5) for RL, MATH500 levels 3-5 for evaluation (disjoint splits)
import random
from pohrl_llm import data as D
SUBJECTS = ['algebra', 'counting_and_probability', 'geometry', 'intermediate_algebra',
            'number_theory', 'prealgebra', 'precalculus']
train = []
for s in SUBJECTS:
    train += D.load_hf('EleutherAI/hendrycks_math', split='train', qkey='problem', akey='solution', config=s,
                       filter_fn=lambda r: str(r.get('level', '')) in ('Level 3', 'Level 4', 'Level 5'))
random.Random(0).shuffle(train); train = train[:N_TRAIN]
for i, r in enumerate(train): r['id'] = f'tr{i}'
ev = [r for r in D.load_hf('math500') if int(r.get('level', 0)) >= 3]
random.Random(1).shuffle(ev); ev = ev[:N_EVAL]
print(len(train), 'train,', len(ev), 'eval')

In [ ]:
# 6. vLLM with LoRA support (shares the GPU with the trainer)
from pohrl_llm.backends import VLLMBackend
from pohrl_llm import prompts as PR
backend = VLLMBackend(MODEL, dtype=DTYPE, gpu_memory_utilization=0.35, enable_lora=True, max_lora_rank=64,
                      max_model_len=6144)
render = lambda msgs: PR.chat(msgs, backend.tokenizer, enable_thinking=False)

In [ ]:
# 7. Plans from the frozen base model, cached on Drive (shared by all methods)
from pohrl_llm import run_m3
import collections
train_planned = run_m3.cache_plans(train, backend, render, f'{OUT}/plans_train.jsonl')
eval_planned = run_m3.cache_plans(ev, backend, render, f'{OUT}/plans_eval.jsonl')
print('train width buckets:', collections.Counter(run_m3.width_bucket(p.plan.poset.width()) for p in train_planned))
print('eval  width buckets:', collections.Counter(run_m3.width_bucket(p.plan.poset.width()) for p in eval_planned))
print('fallback rate:', sum(p.plan.fallback for p in train_planned) / len(train_planned))

In [ ]:
# 8. Train each method (resumable; rerun this cell after a disconnect)
import gc, torch
from pohrl_llm.trainer import LoRATrainer
logs = {}
for seed in SEEDS:
    for m in METHODS:
        factory = lambda run_dir, resume: LoRATrainer(MODEL, run_dir, lr=1e-5, rank=64, resume=resume, micro_bs=4)
        logs[(m, seed)] = run_m3.train(m, seed, train_planned, eval_planned, backend, render, factory, OUT,
                                       steps=STEPS, problems_per_step=PROBLEMS_PER_STEP, G=G,
                                       eval_every=EVAL_EVERY, verify_mode='consensus')
        gc.collect(); torch.cuda.empty_cache()

In [ ]:
# 9. Curves: overall and by width bucket
import json, matplotlib.pyplot as plt
cols = {'flat': '#999', 'chain': '#e08a2c', 'poset': '#2a6fdb'}
fig, ax = plt.subplots(1, 4, figsize=(16, 3.5))
for (m, seed), lg in logs.items():
    ev_rows = [r for r in lg if 'acc' in r]
    for k, key in enumerate(['acc', 'acc_w1', 'acc_w2-3', 'acc_w4+']):
        xs = [r['step'] for r in ev_rows if r.get(key) is not None]
        ax[k].plot(xs, [r[key] for r in ev_rows if r.get(key) is not None], color=cols[m], label=f'{m} s{seed}')
        ax[k].set_title(key); ax[k].set_xlabel('RL step')
ax[0].legend(); plt.tight_layout(); plt.savefig(f'{OUT}/m3_curves.png', dpi=150)

In [ ]:
# 10. Package logs (no adapters) to send back
!cd "{OUT}" && zip -qr ../m3_logs_$(basename "{OUT}").zip */state.json plans_*.jsonl m3_curves.png && ls -la ..